In [ ]:
!pip -q install "transformers==5.0.0" "accelerate>=1.10.1" 2>&1 | tail -3


In [ ]:
import subprocess, sys
probe = subprocess.run([sys.executable, "-c", "from transformers import PreTrainedModel"], text=True, capture_output=True)
if probe.returncode != 0 and "torchvision" in probe.stderr:
    print("removing a torchvision built for another torch")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision"])


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import torch
import transformers

print("torch", torch.__version__, "| transformers", transformers.__version__, "| cpus", os.cpu_count())
print(subprocess.run(["free", "-m"], capture_output=True, text=True).stdout)

found = sorted(Path("/kaggle/input").rglob("repo/scripts/kernel_gates.py"))
assert found, "the dataset has no kernel_gates.py"
sys.path.insert(0, str(found[0].parent))
from kernel_gates import run_bounded  # noqa: E402

os.environ["HF_DEACTIVATE_ASYNC_LOAD"] = "1"  # as in both T4 runs
os.environ["TOKENIZERS_PARALLELISM"] = "false"
PROBE = Path("/kaggle/working/codec_probe.py")
PROBE.write_text("""
import faulthandler, resource, sys, time
faulthandler.dump_traceback_later(60, repeat=True, file=sys.stderr)
started = time.time()
from transformers import AutoModel
codec = AutoModel.from_pretrained(sys.argv[1], trust_remote_code=True, codec_weight_dtype="bf16")
faulthandler.cancel_dump_traceback_later()
params = sum(p.numel() for p in codec.parameters())
print(f"LOADED in {time.time() - started:.0f}s, {params / 1e9:.2f}B parameters, "
      f"peak RSS {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20:.1f} GiB", flush=True)
""")
RESULTS = {}


In [ ]:
from huggingface_hub import snapshot_download

start = time.time()
CODEC_DIR = Path(snapshot_download(repo_id="OpenMOSS-Team/MOSS-Audio-Tokenizer-v2", revision="f6e20e543b33d2c252a7ef71bdf8aa71e5ff9169"))
MODEL_DIR = Path(snapshot_download(repo_id="OpenMOSS-Team/MOSS-TTS-Local-Transformer-v1.5", revision="be7766a6735b98bd793f7c79fb720b4d0f5d13b8"))
print(f"codec and model in {(time.time() - start) / 60:.1f} min")
print("gpus:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


In [ ]:
code, seconds, timed_out = run_bounded([sys.executable, str(PROBE), str(CODEC_DIR)], timeout=600,
                                       drain=True, monitor_every=60, tee=Path("/kaggle/working/drained.log"))
RESULTS["drained_cold_cache"] = {"exit": code, "seconds": round(seconds), "timed_out": timed_out}


In [ ]:
code, seconds, timed_out = run_bounded([sys.executable, str(PROBE), str(CODEC_DIR)], timeout=600,
                                       drain=False, monitor_every=60)
RESULTS["inherited_warm_cache"] = {"exit": code, "seconds": round(seconds), "timed_out": timed_out}


In [ ]:
# The worker's own path: processor (tokenizer + codec), codec to cuda:1, model
# onto cuda:0 in bf16 - then quit as soon as it reports ready. Drained and
# monitored, with a budget that is generous for a load and short for a hang.
scripts = found[0].parent
driver = Path("/kaggle/working/ready_then_quit.py")
driver.write_text(f"""
import json, subprocess, sys, time
start = time.time()
worker = subprocess.Popen([sys.executable, "{scripts}/moss15_worker.py", "--model-dir", sys.argv[1],
                           "--codec-dir", sys.argv[2], "--device", "cuda", "--codec-device", "auto"],
                          stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True)
line = worker.stdout.readline()
print("WORKER SAID", line.strip()[:2000], flush=True)
print(f"READY AFTER {{time.time() - start:.0f}}s", flush=True)
worker.stdin.write(json.dumps({{"op": "quit"}}) + chr(10))
worker.stdin.flush()
worker.wait(timeout=120)
sys.exit(0 if json.loads(line).get("ready") else 1)
""")
code, seconds, timed_out = run_bounded([sys.executable, str(driver), str(MODEL_DIR), str(CODEC_DIR)],
                                       timeout=1200, drain=True, monitor_every=60,
                                       tee=Path("/kaggle/working/worker_load.log"))
RESULTS["worker_path_drained"] = {"exit": code, "seconds": round(seconds), "timed_out": timed_out}


In [ ]:
print(json.dumps(RESULTS, indent=2))
ok = {name: (not r["timed_out"] and r["exit"] == 0) for name, r in RESULTS.items()}
if not ok["drained_cold_cache"]:
    reading = "the codec alone hangs on this image even drained - image-specific; see the wait channels"
elif not ok["inherited_warm_cache"]:
    reading = "codec loads drained but hangs inherited on this image - consistent with an undrained pipe"
elif not ok["worker_path_drained"]:
    reading = "the codec loads alone but the worker's full path hangs - the cause is in that path"
else:
    reading = "every load finished on this image - the T4 kernel's drained, monitored run should get past loading"
print("READING:", reading)
RESULTS["reading"] = reading
Path("/kaggle/working/probe.json").write_text(json.dumps(RESULTS, indent=2))
